# Lab 7 Simple: BERT for Sentiment and Question Answering


## Cell 2: Import Libraries


In [1]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForQuestionAnswering
from transformers import TrainingArguments, Trainer, pipeline


## Part A: Very Basic BERT Sentiment Fine-tuning


## Cell 3: Create Small Sentiment Dataset


In [3]:
texts = [
    "I love this movie",
    "This film is excellent",
    "Amazing story and acting",
    "I hate this movie",
    "This film is boring",
    "Worst acting ever"
]

labels = [1, 1, 1, 0, 0, 0]  # 1 = positive, 0 = negative


## Cell 4: Tokenize Dataset


In [4]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

encodings = tokenizer(texts, truncation=True, padding=True, max_length=64)

class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

dataset = SentimentDataset(encodings, labels)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Cell 5: Fine-tune BERT for Sentiment


In [5]:
sentiment_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir="./simple_bert_sentiment",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=sentiment_model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.696229
2,0.733519
3,0.722018
4,0.681944
5,0.665786
6,0.719491
7,0.683907
8,0.633899
9,0.669328


TrainOutput(global_step=9, training_loss=0.6895691288842095, metrics={'train_runtime': 7.4057, 'train_samples_per_second': 2.431, 'train_steps_per_second': 1.215, 'total_flos': 27942341904.0, 'train_loss': 0.6895691288842095, 'epoch': 3.0})

## Cell 6: Predict Sentiment


In [6]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = sentiment_model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=1).item()
    return "Positive" if prediction == 1 else "Negative"

print(predict_sentiment("I really love this film"))
print(predict_sentiment("This movie is very bad"))


Negative
Negative


## Part B: Very Basic BERT Question Answering


## Cell 7: Load Pretrained QA Model


In [7]:
qa = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Cell 8: Ask a Question


In [8]:
context = "BERT is a transformer model developed by Google. It is used for NLP tasks like sentiment analysis and question answering."
question = "Who developed BERT?"

result = qa(question=question, context=context)
print("Question:", question)
print("Answer:", result["answer"])


Question: Who developed BERT?
Answer: Google
